> 모두 gpt-3.5-turbo 기준으로 작성

## 1. set-up envs

In [1]:
from config import langchain

langchain.setup_langchain_openai_envs()

## 2. set-up data-loaders and splitters

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

character_text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-3.5-turbo",
    chunk_size=600, 
    chunk_overlap=100,
    separator="\n",
)
loader = TextLoader('files/1984_chapter_3_to_6.txt')
docs = loader.load_and_split(text_splitter=character_text_splitter)

## 3. set-up embeddings & vector-store

In [3]:
from langchain_community.vectorstores import Chroma
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_openai import OpenAIEmbeddings

openai_3_small_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
file_store = LocalFileStore('./.cache/')
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=openai_3_small_embeddings,
    document_embedding_cache=file_store
)

vectorstore = Chroma.from_documents(embedding=cached_embeddings, documents=docs)

## 4. set-up Stuff LCEL chain + ConversationBufferMemory

In [4]:
from langchain.memory.chat_memory import BaseChatMemory


class OrdiLlmMemory:
    def __init__(self, memory: BaseChatMemory):
        self.memory = memory
    
    def add_message(self, human_message: str, ai_message: str):
        self.memory.save_context(
            inputs={"human": human_message},
            outputs={"ai": ai_message}
        )
    
    def add_history(self, question, result_content):
        self.memory.save_context(
            inputs={"human": question},
            outputs={"ai": result_content}
        )
    
    def get_history(self, _) -> dict:
        return self.memory.load_memory_variables({})["history"]
    
    
def invoke_chain(target_chain, target_memory, question):
    result = target_chain.invoke(question)
    target_memory.add_history(
        question=question,
        result_content=result.content,
    )
    return result

In [5]:
from langchain.memory import ConversationBufferMemory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)
retriever = vectorstore.as_retriever()
memory = OrdiLlmMemory(memory=ConversationBufferMemory(return_messages=True))

prompt = ChatPromptTemplate.from_messages([
    ('system', "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ('human', '{question}')
])

chain = {
    'context': retriever,
    'history': memory.get_history,
    'question': RunnablePassthrough()
} | prompt | llm

In [6]:
invoke_chain(chain, memory, question="Is Aaronson guilty?")

AIMessage(content='According to the text, Jones, Aaronson, and Rutherford were guilty of the crimes they were charged with.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 2517, 'total_tokens': 2540}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-784113bc-4722-4f24-b179-d1852c81e2a8-0', usage_metadata={'input_tokens': 2517, 'output_tokens': 23, 'total_tokens': 2540})

In [7]:
invoke_chain(chain, memory, question="What message did he write in the table?")

AIMessage(content='Winston traced "2+2=5" with his finger in the dust on the table.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 2534, 'total_tokens': 2554}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-71e1d047-ed8e-4188-8569-55bec5ae8089-0', usage_metadata={'input_tokens': 2534, 'output_tokens': 20, 'total_tokens': 2554})

In [8]:
invoke_chain(chain, memory, question="Who is Julia?")

AIMessage(content='Julia is a character who was involved with Winston in acts of rebellion against the Party.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 2610, 'total_tokens': 2628}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-97a074aa-045c-4fd3-9e81-4c4c1051beef-0', usage_metadata={'input_tokens': 2610, 'output_tokens': 18, 'total_tokens': 2628})